In [ ]:
import pyarrow.parquet as pq
import duckdb
path = "/Users/anna/Desktop/Research/climate-investments/data/raw/builty_all.parquet"
pf = pq.ParquetFile(path)
print(pf.metadata)
#163,332,180 rows

  created_by: DuckDB version v1.4.4 (build 6ddac802ff)
  num_columns: 24
  num_rows: 163332180
  num_row_groups: 1329
  format_version: 1.0
  serialized_size: 5061079


In [2]:
#filter to residential only
var ="PROPERTY_TYPE"

# distinct values + counts
query = f"""
SELECT DISTINCT {var}
FROM read_parquet('{path}')
LIMIT 100
"""

print(duckdb.query(query).to_df())

  PROPERTY_TYPE
0   Residential
1    Commercial
2           NaN


In [ ]:

input_file = path
residential_file = "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_residential.parquet"

duckdb.query(f"""
COPY (
    SELECT *
    FROM read_parquet('{input_file}')
    WHERE PROPERTY_TYPE = 'Residential'
)
TO '{residential_file}' (FORMAT PARQUET)
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
df = pq.ParquetFile(residential_file)
print(df.metadata)
#60,211,670 rows when filtering to residential only

  created_by: DuckDB version v1.4.4 (build 6ddac802ff)
  num_columns: 24
  num_rows: 60211670
  num_row_groups: 486
  format_version: 1.0
  serialized_size: 1932248


In [10]:
#start elevation filtering
#key words to look for
STRICT_PATTERNS = [
    r"elevat",
    r"rais(e|ed|ing)",
    r"lift(ed|ing)",
    r"jack(ed|ing) up",
    r"flood",
    r"fema",
    r"bfe",
    r"base flood elevation",
    r"finished floor elevation",
    r"ffe",
    r"freeboard",
    r"nfip",
    r"icc",
    r"floodplain",
    r"flood plain",
    r"flood zone",
    r"hazard mitigation",
    r"substantial damage",
    r"substantially damaged",
    r"substantial improvement",
    r"substantially improved",
    r"storm surge",
    r"out of (the )?floodplain",
]
#key words to exclude
FALSE_POSITIVE_PATTERNS = [
    r"elevator",
    r"pool",
    r"signage|wall sign|channel letters|illuminated.{0,30}(sign|letter|cabinet)",
    r"tree removal|tree pruning|prun(e|ing)|canopy|oak tree|laurel oak",
    r"rais(e|ed|ing).{0,40}roof|roof.{0,40}rais(e|ed|ing)",
    r"rais(e|ed|ing).{0,40}ceiling|ceiling.{0,40}rais(e|ed|ing)",
    r"raise (the )?roof",
    r"raise (the )?bar",
    r"shower faucet|shower plumbing",
    r"generator",
    r"water heater",
    r"fence permit|privacy fence|rock ?wall",
]

In [14]:
residential_file = "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_residential.parquet"
output_file = "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_loosefilter.parquet"

strict_sql = " OR ".join(
    [f"regexp_matches(desc_l, '{p}')" for p in STRICT_PATTERNS]
)

false_sql = " OR ".join(
    [f"regexp_matches(desc_l, '{p}')" for p in FALSE_POSITIVE_PATTERNS]
)

con = duckdb.connect()
con.execute("SET memory_limit='10GB'")
con.execute("SET threads=4")

query = f"""
COPY (
    WITH prepared AS (
        SELECT
            *,
            lower(coalesce(DESCRIPTION, '')) AS desc_l
        FROM read_parquet('{residential_file}')
    ),

    flagged AS (
        SELECT
            *,
            CASE WHEN ({strict_sql}) THEN 1 ELSE 0 END AS candidate_flag,
            CASE WHEN ({false_sql}) THEN 1 ELSE 0 END AS falsepos_flag
        FROM prepared
    )

    SELECT *
    FROM flagged
    WHERE candidate_flag = 1
      AND falsepos_flag = 0

) TO '{output_file}' (FORMAT PARQUET)
"""

con.execute(query)

print("done")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done


In [ ]:
elevate1 = pq.ParquetFile(output_file)
print(elevate1.metadata)
#744,977 obs


  created_by: DuckDB version v1.4.4 (build 6ddac802ff)
  num_columns: 27
  num_rows: 744977
  num_row_groups: 7
  format_version: 1.0
  serialized_size: 37487


In [20]:
#export to stata
# read parquet
df = duckdb.query(f"""
SELECT *
FROM read_parquet('{output_file}')
""").to_df()

# write stata file
df.to_stata(
    "/Users/anna/Desktop/Research/climate-investments/data/clean/temp/builty_loosefilter.dta",
    write_index=False,
    version=118
)


## Pipeline summary — observation counts

| Step | File | Obs | Notes |
|------|------|-----|-------|
| Raw | `data/raw/builty_all.parquet` | 163,332,180 | all property types |
| Residential filter | `data/clean/temp/builty_residential.parquet` | 60,211,670 | `PROPERTY_TYPE == 'Residential'` |
| Loose keyword filter | `data/clean/temp/builty_loosefilter.parquet` | 744,977 | candidate=1, falsepos=0 |
| Export to Stata | `data/clean/temp/builty_loosefilter.dta` | 744,977 | — |
| **New records — genuine** | `data/clean/temp/builty_new_genuine.dta` | ~9,671 | `2_filter_new.do` |
| **Non-New records — filtered** | `data/clean/temp/builty_nonnew_filtered.dta` | ~9,800 | `3_filter_nonnew.do` |

### Stata do-files (next steps after this notebook)

**`2_filter_new.do`**  
Loads `builty_loosefilter.dta`, keeps `WORK_TYPES == "New"` (280,198 obs), drops "flood plain: no" boilerplate, applies genuine flood-adaptation signal requirement. Expected output: ~9,671 genuine new-construction flood-elevation cases.

**`3_filter_nonnew.do`**  
Loads `builty_loosefilter.dta`, drops `WORK_TYPES == "New"` (464,779 remain), applies boilerplate exclusions + additional non-New FP exclusions (raised deck/porch, flood lights, flood damage repair, no-elev-cert records, Zone X, HVAC same-elevation), then applies genuine signal requirement. Expected output: ~9,800 genuine non-New flood-adaptation cases.

### Why so few survive?
The loose filter's main false positive drivers:
- `elevat` → "plan elevation: A/B/C" builder façade labels (new construction)
- `flood` → "flood plain: no" standard checklist boilerplate on all new builds
- `rais` → raised decks, patio additions, electrical mast raising
- `elevat` + `flood` in same record but far apart / unrelated context

In [18]:
#check tx and va remaining amount
duckdb.query(f"""
SELECT STATE, COUNT(*) AS n
FROM read_parquet('{output_file}')
WHERE upper(STATE) IN ('TX', 'VA')
GROUP BY STATE
ORDER BY STATE
""").to_df()

,STATE,n
0,TX,68819
1,VA,41475
